In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!pip install -U langchain
#!pip install -U langchain-text_splitter
#!pip install -U langchain-chroma
#!pip install -U langchain-groq
#!pip install -U Sentence_transformers
#!pip install -U transformers
#!pip install -U torch
#!pip install -U Unstructured
#!pip install -U classic pypdf
#!pip install -U langchain-community

In [3]:

import os

# PDF Document Loader
from langchain_community.document_loaders import PyPDFLoader

# Text Splitting
from langchain_text_splitters import CharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector Database
from langchain_chroma import Chroma

# Groq LLM
from langchain_groq import ChatGroq

# Retrieval QA
from langchain_classic.chains import RetrievalQA

In [8]:
# import PDF from the URL

import requests

url = "https://rbberger.github.io/assets/files/smr2574/OOPConcepts.pdf"
response = requests.get(url)

In [9]:
# Saving the PDF to local drive

with open("OOPSConcepts.pdf", "wb")as f:
  f.write(response.content)

In [ ]:
# Loading PDf Document

loader = PyPDFLoader("OOPSConcepts.pdf")
document = loader.load()
document

In [16]:
# Spliting the pdf into smaller chuncks for emedding and retrieval

text_splitter = CharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 400
)

texts = text_splitter.split_documents(document)


In [ ]:
#Embedding for document chunks

embedding = HuggingFaceEmbeddings()
persist_directory = "vector_db"

In [45]:
# Storing all embedded data into vector database

vectordb = Chroma.from_documents(
    documents = texts,
    embedding = embedding,
    persist_directory = persist_directory
)

In [46]:
# Converting the Embedded data to retrieval data

retriever = vectordb.as_retriever()

In [ ]:
from getpass import getpass

#Enter your Groq API Key

os.environ["GROQ_API_KEY"] = getpass("Enter the Groq API Key")

In [ ]:
# Create LLM model Groq

llm = ChatGroq(
    model = "qwen/qwen3.8-27b",
    temperature = 0
)

In [ ]:
# Now creating RAG (Retrieval Augmentation Generation) Question and Answering chain

qa_chain = RetrievalQA.from_chain_type(
     llm = llm,
    chain_type = "stuff",
    retriever = retriever,
    return_source_documents = True
)

In [ ]:
# Now invoke Question and Answer chain for the user Query

query = "What are functions mentioned in PDF"

response = qa_chain.inove({"query": query})

print(response["result"])

In [ ]:
#!pip install -U gradio

In [31]:
def ask_question(query):
  response = qa_chain.invoke({
      "query": query
  })

  answer = response["result"]  # Get generated answer

  source = response["source_documents"][0].metadata["source"] # Get the source document

  return answer, source

In [50]:
import gradio as gr

def ask_question(query):
    response = qa_chain.invoke({"query": query})
    return response["result"]

with gr.Blocks() as demo:

    gr.Markdown("Document Intelligence System using RAG")
    gr.Markdown("Ask questions about the OOPConcepts PDF.")

    query = gr.Textbox(
        label="Ask a Question",
        placeholder="What is polymorphism?"
    )

    submit = gr.Button("Submit")

    answer = gr.Textbox(
        label="Answer",
        lines=8
    )

    submit.click(
        fn=ask_question,
        inputs=query,
        outputs=answer
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43c8f79de9c489135c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
